# Lesson 11 : Hosted Agents in Microsoft Foundry

When you deploy and run your generated agent's code for production, you can, of course, host your agent by yourself.  
However, if you're working on Microsoft Foundry, you can host your agent by publishing into Microsoft Foundry as a **hosted agent**.

It runs in a micro-VM sandbox (backed by **Azure Container Apps Sandboxes**), which provides CPU and memory allocation to run the agent.<br>
The hosted agent platform in Microsoft Foundry provides a fully managed runtime to deploy and operate it at enterprise scale - with security, isolation (hypervisor-isolated sandbox), and performance that production workloads demand. (Without changing any code, you can gain faster startup, zero idle cost, virtual network support, observability, built-in management functionalities, guardrails, Entra ID protection, and more.)

The code built by Microsoft Agent Framework, LangChain, and custom code can be run as a hosted agent. (Workflow agents in Microsoft Agent Framework can also work as hosted agents.)

Foundry hosted agents can be used and run in the following three ways :

- Assistive agents (chat agents) : act interactively on the user's behalf inside conversation (including voice live chat agent)
- Autonomous agents (routines) : act on their own behalf in the background by triggers (scheduling)
- Autopilot agents : act autonomously like humans (also long-running - such as, all day long) using their own identity and have user accounts with a productivity license granting them their own email, calendar, OneDrive, Microsoft Teams access

> Note : For running autopilot agents as Foundry hosted agents, use Agent 365 SDK. Please see [here](https://github.com/microsoft-foundry/foundry-samples/tree/main/samples/python/foundry-autopilot-agent) for this sample code.

There exist the following 2 types of deployment :

- Container-based hosted agent : The asset for deployment is a container image.
- Source-code-based hosted agent : The asset for deployment is a .zip file including code. (The source-code deployments simplifies CI/CD and keeps the deployment flexible - such as, integrating with GitHub Actions.)

In this exercise, we deploy and run an agent (written by Microsoft Agent Framework) as Foundry hosted agent in the following configuration :

- Deploy with a container image
- Run as a simple chat agent (assistive agent)

## 1. Prepare environment

To start this exercise, you should install ```azd``` command (Azure Developer CLI) on your working environment.

> Note : Here we use ```azd``` command to configure infrastructure, but you can also manually provision Azure infrastructure and deploy your hosted agent with Python SDK.

After installation, login to Azure in ```azd``` by running the following command.

```
azd auth login --use-device-code
```

Next, install ai agent extension (```azure.ai.agents```) in ```azd``` by running the following command.

```
azd extension install azure.ai.agents
```

Finally, install docker, because hosted agents is registered as container.  
In my case, I have used the following command to install docker in Ubuntu 24.04, and logout/login after installation.

```
# update apt
sudo apt-get -y update
# renew ca-certification and related packages
sudo apt-get -y install apt-transport-https ca-certificates curl software-properties-common
# download and run gpg for secure communication
curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg
# add repository
echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
# update apt with new repository
sudo apt-get -y update
# display properties of docker-ce
apt-cache policy docker-ce
# install docker ce from repository
sudo apt-get -y install docker-ce
# update user permission
# Important! : logout / login to take effect after update
sudo usermod -aG docker $USER
```

## 2. Prepare resources in Azure

Please create a Foundry resource **in the region where hosted agents are supported**.  
For the supported regions by the hosted agent, please see [here](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents#region-availability).

> Note : In this exercise, we prepare Foundry resource in advance, but you can also create Foundry resource in the following provisioning phase, without creating manually by yourself.

In Foundry Portal, deploy an Azure OpenAI model, which is supported in Azure OpenAI Responses API. (See [here](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry&tabs=python-key#model-support) for the supported models.)  
In the following setting, we assume we have deployed with deployment name "**gpt-5**".

In hosted agents, other resources (such as, container registry, etc) are also required. In this example, however, these other resources are automatically created by running the following provisioning steps. (No other resources, therefore, should be prepared manually.)

## 3. Prepare files

As I have mentioned above, we'll apply container-based deployment in this exercise.<br>
In order for provisioning as a hosted agent, therefore, you should prepare the following files in this example.

- Python source code (main.py)
- Python package list (requirements.txt)
- Docker file (Dockerfile)
- Agent configuration (agent.yaml)
- Agent manifest (agent.manifest.yaml)

In this example, we'll store these files in ```hosted_agent_work``` folder. Hence we create this folder as follows at first.

In [1]:
import os
os.makedirs("hosted_agent_work", exist_ok=True)

First we prepare Python source code file (```main.py```).

Foundry hosted agents support 2 types of protocols, **Open Responses protocol** or **Invocation protocol**.  
Since we'll use Open Responses protocol in this example, the agent server is abstracted by adapter function, ```ResponsesHostServer()```, as follows.

It's worth noting that, unlike other examples run locally, **the credential returned by ```DefaultAzureCredential``` in hosted agent is not the user's credential, but the app credential of the Agent identity** that identity holds "Foundry User" role for the Foundry project. (If a user token is required, the Foundry platform internally restores the user token using the token passed by the caller.)<br>
See [here](https://tsmatz.wordpress.com/2026/03/12/microsoft-agent-identity-for-developers/) for details about Agent identity.

Please note that the ```%%writefile``` directive (the first line in the following code) in Jupyter notebook cell indicates that this code is stored as file, not executed here. (The same applies in the following cells.)

> Note : The hosting libraries used in hosted agents (```azure-ai-agentserver-responses``` and ```azure-ai-agentserver-invocations```) smoothly integrate with Microsoft OpenTelemetry distro library (```microsoft-opentelemetry```). Your agent, therefore, automatically emits traces as seen in [Lesson 2](./02_trace.ipynb), when enabled tracing in your Foundry project.

In [2]:
%%writefile hosted_agent_work/main.py
import os
from typing import Annotated
from pydantic import Field
from random import randint
from agent_framework.foundry import FoundryChatClient, ResponsesHostServer
from agent_framework import Agent, tool
from azure.identity import DefaultAzureCredential

@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="the location to get the weather for")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]}."

@tool(approval_mode="never_require")
def get_temperature(
    location: Annotated[str, Field(description="the location to get the temperature for")],
) -> str:
    """Get the temperature for a given location."""
    return f"The temperature in {location} is {randint(10, 30)} degrees."

if __name__ == "__main__":
    credential = DefaultAzureCredential()
    client = FoundryChatClient(
        project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
        model=os.environ["MODEL_DEPLOYMENT_NAME"],
        credential=credential,
    )
    agent = Agent(
        client=client,
        instructions="You are an agent about weather information.",
        tools=[get_weather, get_temperature],
        default_options={"store": False},
    )
    server = ResponsesHostServer(agent)
    server.run()

Writing hosted_agent_work/script/main.py


Next we prepare ```requirements.txt```, a list of Python packages, which are installed on docker image generation.

To use adapter for hosted agent (i.e., ```ResponsesHostServer()``` in above code), we should also install Python package ```agent-framework-foundry-hosting```.

In [3]:
%%writefile hosted_agent_work/requirements.txt
agent-framework
agent-framework-foundry-hosting

Writing hosted_agent_work/script/requirements.txt


Now we create docker file, ```Dockerfile```.  
The above ```ResponsesHostServer()``` automatically creates a REST endpoint by port 8088 and we then expose this port in docker image generation as follows.

In [4]:
%%writefile hosted_agent_work/Dockerfile
FROM python:3.12-slim

WORKDIR /app

COPY . user_agent/
WORKDIR /app/user_agent

RUN if [ -f requirements.txt ]; then \
        pip install -r requirements.txt; \
    else \
        echo "No requirements.txt found"; \
    fi

EXPOSE 8088

CMD ["python", "main.py"]

Writing hosted_agent_work/script/Dockerfile


The file, ```agent.yaml```, includes the information about resources running the hosted agent - such as, CPU allocation, memory allocation.  
On contrary, the file, ```agent.manifest.yaml```, includes the agent's information - such as, agent name, the required environment variables.  
These settings are used in the following provisioning phase by ```azd``` command execution.

As I have mentioned above, we use Open Responses protocol in this example and we then set ```responses``` as ```protocol``` property as follows.

We use ```FoundryChatClient``` in our code, which requires ```FOUNDRY_PROJECT_ENDPOINT``` and ```MODEL_DEPLOYMENT_NAME``` environment variables.  
The hosted agent platform automatically injects environment variable ```FOUNDRY_PROJECT_ENDPOINT```. Therefore, in this case, we only set ```MODEL_DEPLOYMENT_NAME``` environment variable manually as follows.

Please note that **you should replace the following "```gpt-5```" with your deployment name.**

In [5]:
%%writefile hosted_agent_work/agent.yaml
kind: hosted
name: hosted-test-agent01
protocols:
  - protocol: responses
    version: 2.0.0
resources:
  cpu: "0.25"
  memory: "0.5Gi"
environment_variables:
  - name: MODEL_DEPLOYMENT_NAME
    value: ${MODEL_DEPLOYMENT_NAME}

Writing hosted_agent_work/script/agent.yaml


In [6]:
%%writefile hosted_agent_work/agent.manifest.yaml
name: hosted-test-agent01
description: This is a hosted agent for demo.
metadata:
  authors:
    - Workshop demo
template:
  name: hosted-test-agent01
  kind: hosted
  protocols:
    - protocol: responses
      version: 2.0.0
  environment_variables:
    - name: MODEL_DEPLOYMENT_NAME
      value: "{{chat_model}}"
resources:
  - kind: model
    id: gpt-5
    name: chat_model

Writing hosted_agent_work/agent.manifest.yaml


## 4. Provision, deploy, and run

Now the assets are all ready.  
We now start to configure, provision Azure infrastructure, and deploy your hosted agent, with ```azd``` commands.

**All the following settings should be performed in console (terminal)**, not in Jupyter notebook. (Because Jupyter notebook cannot handle the interactive session.)

Before starting, let's change your working directory to subfolder ```hosted_agent_work```. (Because the execution commands automatically detect the files in your current directory.)

```bash
cd hosted_agent_work
```

---

Run the following command to prepare infrastructure setting.  
Before runnning, **please change the following placeholders** in the following command. To get string for ```--project-id``` option, go to foundry project resource on [Azure Portal](https://portal.azure.com/), open "Resource Management" - "properties" in left-side navigation, and you then find it in "Resource ID".

```bash
azd ai agent init --project-id /subscriptions/[SUBSCRIPTION-ID]/resourceGroups/[RESOURCE-GROUP-NAME]/providers/Microsoft.CognitiveServices/accounts/[FOUNDRY-RESOURCE-NAME]/projects/[PROJECT-NAME]
```

During running this command, you will be asked to specify existing Azure Container Registry (ACR) server name and Application Insights connection name. But **please set blank**, and the hosted agent platform will then automatically create these resources in provisioning steps.

> Note : If you have already connected to these resources (app insights, container registry) in your Foundry resource, do not set blank, because multiple connection to the same category are not allowed in Microsoft Foundry. (The error will then be thrown.)

You will be asked for the deployment method - code or container image -, and please select "**Container Image (Docker)**".

You will also be asked for the following settings, but here we apply the default settings as follows.  
(These provisioning definitions are then written in azure.yaml.)

- CPU allocation : 0.25 cores
- Memory allocation : 0.5 Gi

By running this command, the required configurations (bicep configurations, etc) are setup in "```infra```" folder. The environment variables (such as, ```FOUNDRY_PROJECT_ENDPOINT```, ```MODEL_DEPLOYMENT_NAME```, etc) are automatically associated to your setting in the configuration.

---

**[Optional] This is not mandatory for running this exercise.**

When you run and test your AI agent locally, follow this step.

First, please set environment variables for running your agent. In this example, we should set ```FOUNDRY_PROJECT_ENDPOINT``` and ```MODEL_DEPLOYMENT_NAME```.

```
export FOUNDRY_PROJECT_ENDPOINT=[AZURE-OPENAI-ENDPOINT]
export MODEL_DEPLOYMENT_NAME=gpt-5
```

Before running, generate new virtual environment in Python and activate. (Because it installs packages written in ```requirements.txt```.)

Run your agent locally by running the following command.

```
azd ai agent run hosted-test-agent01
```

Open another console, and send a message by running the following command.

```
azd ai agent invoke "Tell me the weather and temperature in Osaka today." --local
```

> Note : **Do not enable managed identity** in your working VM. If it's enabled, the managed identity will be used for your current credential (in ```DefaultAzureCredential``` in your code) and it will then fail.

---

Now let's provision the infrastructure by running the following command.  
After running this command, all required resources - Application Insights, Log Analytics workspace, and Azure Container Registry (ACR) - are provisioned in the same resource group. (Also, the container image is built and registered in ACR.)

```bash
azd provision
```

---

Now all Azure resources are ready.  
Finally we deploy and start our hosted agent in Microsoft Foundry by running the following command.

```bash
azd deploy
```

> Note :  By running ```azd up```, provisioning (```azd provision```) and deployment (```azd deploy```) are both performed.  
> By running ```azd down```, it cleans up all resources in resource group.  
> By using ```azd up``` / ```azd down```, you can quickly build all resources and then clean up. (It's very useful in development.)

After the hosted agent is successfully deployed, you can see your agent running in Foundry Portal UI. (The hosted agent is labeled "```hosted```" as type.)

## 5. Consume your hosted agent

To consume your hosted agent, you can use Playground in Foundry Portal UI, or invoke API.  
Since the hosted agent in this example can talk with Open Responses protocol, you can use Azure AI Projects SDK (```azure-ai-projects```), OpenAI SDK, or raw REST in this case.  
The following code extracts generic ```OpenAI``` client by using Azure AI Projects SDK, and invokes the request with this client. (Before running this cell, **replace the placeholders with your Foundry setting**, in which your hosted agent is running.)

In [7]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from IPython.display import Markdown, display

AGENT_NAME = "hosted-test-agent01"
PROJECT_ENDPOINT = "https://[FOUNDRY-NAME].services.ai.azure.com/api/projects/[FOUNDRY-PROJECT-NAME]"

# initialize the client and retrieve the agent
project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=AzureCliCredential(),
    allow_preview=True,
)

# get OpenAI client and send a message
# (it's OpenAI Responses API.)
openai_client = project_client.get_openai_client(agent_name=AGENT_NAME)
conversation = openai_client.conversations.create()
response = openai_client.responses.create(
    conversation=conversation.id,
    input=[{"role": "user", "content": "Tell me the weather and temperature in Osaka today."}],
)

# show result
display(Markdown(response.output_text))

Osaka, Japan: **Cloudy**, **11 °C**.

## 6. Use your hosted agent in Microsoft ecosystem (Publish as Microsoft 365 Copilot Agent)

Once it's deployed as hosted agent, your agent can also be consumed in Microsoft ecosystem frameworks.

For example, Microsoft 365 Copilot supports **Activity Protocol**, and your hosted agent can be exposed an agent that supports this protocol with one-click deployment experience in Foundry portal UI. (See below picture.)

![Publish to Microsoft 365](./assets/publish_m365.png)